In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/home/dongmin/userdata/open-score-string-quartets


/home/dongmin/.local/share/virtualenvs/open-score-string-quartets-wd2Cnojv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import sys
import subprocess
import warnings
from typing import Union, Any, Optional
import shutil
from pathlib import Path
from collections import Counter, defaultdict
from operator import itemgetter, contains, eq # contains(A, B) == (B in A), eq(A, B) == (A == B)
from itertools import groupby
from tempfile import NamedTemporaryFile, TemporaryDirectory

import math
import random

import json
import csv
import strictyaml as syaml

from tqdm import tqdm
import matplotlib.pyplot as plt

import cv2
import numpy as np
import music21

import xml.etree.ElementTree as ET
from modules.lmxe.lmxe import linearize_lmxe, delinearize_lmxe
from modules.lmxe.lmxe import split_tokens_by_key_token
from modules.lmxe.lmxe.Linearizer import SQLinearizer
from modules.lmxe.lmxe.Delinearizer import SQDelinearizer
from modules.lmxe.lmxe.LMXEFile import LMXEFile, LMXEMetadata, get_score_type
from modules.lmxe.lmxe.symbolic.MxlFile import MusicXMLFile
from modules.lmxe.lmxe.symbolic.part_to_score import parts_to_score

from scripts.mscx_utils import render_lmx
from scripts.utils import get_ts, PathLike, dformat, dprint
from scripts.utils import crop_white_space, load_ossq_metadata

from IPython.core.magic import register_cell_magic
@register_cell_magic
def skip(line, cell):
    return

In [3]:
data_dir = Path.home() / 'userdata' / 'olimpic_dataset_yolo' / 'grandstaff-lmx'

## Make full metadata

In [4]:
%%skip

splits = ['train', 'valid', 'test']
full_data = []
for sn in splits:
  with open(data_dir / f'{sn}-segments.csv', 'r') as f:  # skip header
    d = csv.reader(f)
    d = list(d)[1:]
  full_data += [ r for r in d ]

In [5]:
%%skip

full_data[:3]

In [6]:
%%skip

filtered_data = []

for composer, collection, piece, segment, lmx_length, image_width in full_data:
  p = data_dir / composer / collection / piece / f'{segment}.musicxml'
  if p.exists():
    filtered_data.append((composer, collection, piece, segment, lmx_length, image_width))

In [7]:
%%skip
all([
  (data_dir / composer / collection / piece / f'{segment}.musicxml').exists()
  for composer, collection, piece, segment, lmx_length, image_width in filtered_data
])

In [8]:
%%skip
with open(data_dir / 'full-segments.csv', 'w') as f:
  w = csv.writer(f)
  w.writerow(['composer', 'collection', 'piece', 'segment', 'lmx_length', 'image_width'])
  w.writerows(filtered_data)

## Load metadata

In [4]:
with open(data_dir / 'full-segments.csv', 'r') as f:
  gs_metadata = csv.reader(f)
  header = next(gs_metadata)  # skip header
  gs_metadata = list(gs_metadata)

print(len(gs_metadata), 'segments')
print(header)
print(json.dumps(gs_metadata[:3], indent=2))

53858 segments
['composer', 'collection', 'piece', 'segment', 'lmx_length', 'image_width']
[
  [
    "joplin",
    "joplin",
    "newrag",
    "original_m-106-109",
    "231",
    "338"
  ],
  [
    "joplin",
    "joplin",
    "newrag",
    "min3_up_m-85-88",
    "248",
    "374"
  ],
  [
    "joplin",
    "joplin",
    "newrag",
    "min3_up_m-100-103",
    "255",
    "315"
  ]
]


In [5]:
gs_metadata = [
  data_dir / r[0] / r[1] / r[2] / f"{r[3]}.musicxml"
  for r in gs_metadata
]

all([
  p.exists() for p in gs_metadata
])

True

## MusicXML => LMXE => MusicXML Verification

In [9]:
from functools import partial
from scripts.utils.lmxe_integrity import compare_note_sequences, diagnose_semitone_errors, find_missing_notes, comprehensive_accidental_diagnosis, diagnose_chord_differences, comprehensive_timing_analysis

diagnose_functions={
  'compare': compare_note_sequences,
  'pitch': diagnose_semitone_errors,
  'chord': diagnose_chord_differences,
  'offset': partial(comprehensive_timing_analysis, tolerance=0.002),
  'differs': partial(find_missing_notes),
  'accidental': partial(comprehensive_accidental_diagnosis),
}

In [10]:
with open('/home/dongmin/userdata/olimpic_dataset_yolo/grandstaff-lmx/verify_errors_grandol_2025-10-12-17:40:05.log', 'r') as f:
  errors = f.readlines()

errors = [ l.rstrip().split(': ') for l in errors if l.rstrip() ]
print(len(errors), 'samples')
errors = [ l for l in errors if l[0] == '[ER]' ]

print(len(errors), 'errors')

53858 samples
194 errors


In [11]:
len(errors)

194

In [12]:
errors[:2]

[['[ER]',
  'beethoven/piano-sonatas/sonata16-2/min3_up_m-56-61',
  'Note 1',
  'pitch differs 69 vs 68'],
 ['[ER]',
  'beethoven/piano-sonatas/sonata16-2/original_m-56-61',
  'Note 1',
  'pitch differs 66 vs 65']]

In [13]:
errors_by_type = defaultdict(list)

for _, segment, position, error in errors:
  if 'type' in error:
    errors_by_type['type'].append((segment, position))
  elif 'pitch' in error:
    errors_by_type['pitch'].append((segment, position))
  elif 'offset' in error:
    errors_by_type['offset'].append((segment, position))
  elif 'duration' in error:
    errors_by_type['duration'].append((segment, position))
  elif 'pitches' in error:
    errors_by_type['chord'].append((segment, position))
  elif 'count' in position:
    errors_by_type['count'].append((segment, position))

In [14]:
for etype, errs in errors_by_type.items():
  print(f'{etype}: {len(errs)} errors')

pitch: 122 errors
count: 68 errors
duration: 4 errors


In [15]:
error_type = 'pitch'

In [16]:
errors = errors_by_type[error_type]
errors

[('beethoven/piano-sonatas/sonata16-2/min3_up_m-56-61', 'Note 1'),
 ('beethoven/piano-sonatas/sonata16-2/original_m-56-61', 'Note 1'),
 ('beethoven/piano-sonatas/sonata16-2/maj2_up_m-56-61', 'Note 1'),
 ('chopin/mazurkas/mazurka50-3/maj3_up_m-100-105', 'Note 1'),
 ('beethoven/piano-sonatas/sonata30-3/maj3_up_m-154-157', 'Note 1'),
 ('beethoven/piano-sonatas/sonata30-3/maj3_up_m-187-190', 'Note 1'),
 ('beethoven/piano-sonatas/sonata12-1/maj3_down_m-160-164', 'Chord 1'),
 ('scarlatti-d/keyboard-sonatas/L056K281/min3_up_m-56-61', 'Note 1'),
 ('beethoven/piano-sonatas/sonata04-3/original_m-58-61', 'Note 1'),
 ('beethoven/piano-sonatas/sonata04-3/maj2_down_m-57-61', 'Note 1'),
 ('beethoven/piano-sonatas/sonata04-3/maj3_down_m-57-61', 'Note 1'),
 ('beethoven/piano-sonatas/sonata04-3/min3_up_m-57-61', 'Note 1'),
 ('beethoven/piano-sonatas/sonata04-3/maj2_up_m-58-61', 'Note 1'),
 ('beethoven/piano-sonatas/sonata21-4/maj2_up_m-97-101', 'Note 1'),
 ('beethoven/piano-sonatas/sonata21-4/maj3_up_m-

In [17]:
errors = [error for error in errors if error[1] != 'Note 1']
len(errors)

36

In [20]:
errors

[('beethoven/piano-sonatas/sonata12-1/maj3_down_m-160-164', 'Chord 1'),
 ('scarlatti-d/keyboard-sonatas/L348K244/min3_down_m-121-124', 'Chord 1'),
 ('scarlatti-d/keyboard-sonatas/L348K244/maj3_up_m-121-126', 'Chord 1'),
 ('chopin/mazurkas/mazurka30-1/maj2_up_m-25-30', 'Chord 1'),
 ('mozart/piano-sonatas/sonata08-1/maj3_down_m-91-96', 'Note 93'),
 ('mozart/piano-sonatas/sonata08-1/maj2_up_m-94-97', 'Note 19'),
 ('mozart/piano-sonatas/sonata08-1/maj2_down_m-91-96', 'Note 93'),
 ('mozart/piano-sonatas/sonata08-1/maj3_up_m-93-97', 'Note 42'),
 ('mozart/piano-sonatas/sonata08-1/min3_down_m-93-97', 'Note 42'),
 ('mozart/piano-sonatas/sonata08-1/original_m-91-96', 'Note 93'),
 ('mozart/piano-sonatas/sonata08-1/min3_up_m-93-97', 'Note 43'),
 ('beethoven/piano-sonatas/sonata29-4/original_m-10-13', 'Chord 42'),
 ('beethoven/piano-sonatas/sonata29-4/min3_up_m-149-153', 'Note 60'),
 ('beethoven/piano-sonatas/sonata31-3/maj3_up_m-7-10', 'Note 5'),
 ('beethoven/piano-sonatas/sonata31-3/maj2_down_m-6

In [18]:
error_idx = 0

In [21]:
err = errors[error_idx]
print(err)
segment, _ = err
base_path = data_dir / segment
compare_note_sequences(base_path.with_suffix('.pruned'), base_path.with_suffix('.delin'))

('beethoven/piano-sonatas/sonata12-1/maj3_down_m-160-164', 'Chord 1')


(False, 'Chord 1: pitches differ (49, 52, 58) vs (50, 53, 58)')

In [23]:
diag_type = 'accidental'
diag_type = 'offset'
diag_type = 'differs'
diag_type = 'pitch'
diag_type = 'chord'

diagnose_functions[diag_type](
  base_path.with_suffix('.pruned'),
  base_path.with_suffix('.delin')
)

=== CHORD STRUCTURE ANALYSIS ===
Original: 18 chords, 23 single notes
Decoded:  18 chords, 23 single notes

Chord structure differences: 1
  Type changes (note ↔ chord): 0
  Pitch content changes: 1

=== DETAILED CHORD DIFFERENCES (first 10) ===

Element 0 in Measure 1:
  Original: chord - ('B-', 'D-', 'F-') (3 notes)
  Decoded:  chord - ('B-', 'D', 'F') (3 notes)
  Changes: Added pitches: ['D', 'F']; Removed pitches: ['C#', 'E']


/home/dongmin/.local/share/virtualenvs/open-score-string-quartets-wd2Cnojv/lib/python3.10/site-packages/music21/stream/base.py:3675: Music21DeprecationWarning: .flat is deprecated.  Call .flatten() instead
  return self.iter().getElementsByClass(classFilterList)


[{'index': 0,
  'comparison': {'identical': False,
   'differences': ["Added pitches: ['D', 'F']",
    "Removed pitches: ['C#', 'E']"],
   'info1': {'type': 'chord',
    'pitches': (49, 52, 58),
    'pitch_names': ('B-', 'D-', 'F-'),
    'note_count': 3,
    'root': 'B-',
    'bass': 'D-',
    'duration': 0.5,
    'offset': 0.0,
    'is_grace': True},
   'info2': {'type': 'chord',
    'pitches': (50, 53, 58),
    'pitch_names': ('B-', 'D', 'F'),
    'note_count': 3,
    'root': 'B-',
    'bass': 'D',
    'duration': 0.5,
    'offset': 0.0,
    'is_grace': True}},
  'measure1': 1,
  'measure2': 1}]

## add post processing to krn->musicxml pipeline

In [15]:
with open(data_dir / 'full-segments.csv', 'r') as f:
  gs_metadata = csv.reader(f)
  header = next(gs_metadata)  # skip header
  gs_metadata = list(gs_metadata)

print(len(gs_metadata), 'segments')
print(header)
print(json.dumps(gs_metadata[:3], indent=2))

53858 segments
['composer', 'collection', 'piece', 'segment', 'lmx_length', 'image_width']
[
  [
    "joplin",
    "joplin",
    "newrag",
    "original_m-106-109",
    "231",
    "338"
  ],
  [
    "joplin",
    "joplin",
    "newrag",
    "min3_up_m-85-88",
    "248",
    "374"
  ],
  [
    "joplin",
    "joplin",
    "newrag",
    "min3_up_m-100-103",
    "255",
    "315"
  ]
]


### fix voice counting logic 

In [16]:
sample = gs_metadata[0]
sample_path = data_dir / '/'.join(sample[:4])
sample_path = sample_path.with_suffix('.musicxml')
print(sample_path)

# Parse the XML file
tree = ET.parse(sample_path)
root = tree.getroot()

# Find all <note> elements that have <staff>2</staff> as a child
notes = root.findall(".//note[staff='2']")

print(len(notes))

# Iterate through the results
for note in notes:
  print(note)
  # You can access the note's attributes and children here

/home/dongmin/userdata/olimpic_dataset_yolo/grandstaff-lmx/joplin/joplin/newrag/original_m-106-109.musicxml
22
<Element 'note' at 0x7760fbfa2750>
<Element 'note' at 0x7760fbfa2a20>
<Element 'note' at 0x7760fbfa2cf0>
<Element 'note' at 0x7760fbfa2fc0>
<Element 'note' at 0x7760fbfa3290>
<Element 'note' at 0x7760fbfa3560>
<Element 'note' at 0x7760fbfa3830>
<Element 'note' at 0x7760fbfa3b00>
<Element 'note' at 0x7760fbf8dc10>
<Element 'note' at 0x7760fbf8de90>
<Element 'note' at 0x7760fbf8e160>
<Element 'note' at 0x7760fbf8e430>
<Element 'note' at 0x7760fbf8e700>
<Element 'note' at 0x7760fbf8e8e0>
<Element 'note' at 0x7760fbe5c5e0>
<Element 'note' at 0x7760fbe5e2a0>
<Element 'note' at 0x7760fbe5e570>
<Element 'note' at 0x7760fbe5e890>
<Element 'note' at 0x7760fbe5eca0>
<Element 'note' at 0x7760fbe5f0b0>
<Element 'note' at 0x7760fbe5f3d0>
<Element 'note' at 0x7760fbe5f6a0>
